In [1]:
from __future__ import annotations

import argparse
from dataclasses import dataclass
from typing import List, Optional

import gymnasium as gym
from pystk2_gymnasium import AgentSpec
from pystk2_gymnasium.definitions import CameraMode
import pystk2_gymnasium
import pystk2

In [2]:

SUPPORTED_ENV_IDS = {
    "supertuxkart/full-v0",
    "supertuxkart/simple-v0",
    "supertuxkart/flattened-v0",
    "supertuxkart/flattened_continuous_actions-v0",
    "supertuxkart/flattened_multidiscrete-v0",
    "supertuxkart/flattened_discrete-v0",
}
@dataclass
class SuperTuxKartEnvConfig:
    """
    Configuration container for `make_supertux_env`.

    Parameters
    ----------
    env_id : str
        The Gymnasium registry ID for the desired SuperTuxKart environment.
        Must be one of `SUPPORTED_ENV_IDS`.

    render_mode : str or None
        Rendering strategy ("human", "rgb_array") or None for headless mode.

    track : str or None
        Name of the racing track. If None, a random one is selected.

    num_kart : int
        Number of karts spawned in the race.

    laps : int
        Number of laps required to finish the episode.

    difficulty : int
        Built-in AI difficulty level (0 = Easy, 2 = Hard).

    max_paths : int or None
        If supported by the environment, controls the number of forecasted path
        nodes used in observations.

    agent_rank_start : int or None
        Starting position of the controlled kart (None → random assignment).

    agent_name : str
        Display name for the agent.

    agent_use_ai : bool
        If `True`, the internal SuperTuxKart AI controls the kart instead of RL actions.

    agent_camera_mode : CameraMode
        Camera configuration for the controlled kart.

    Example
    -------
    >>> config = SuperTuxKartEnvConfig(render_mode="human", laps=2, agent_name="MyModel")
    >>> env = make_supertux_env(config)
    """

    env_id: str = "supertuxkart/full-v0"
    render_mode: Optional[str] = None
    track: Optional[str] = None
    num_kart: int = 4
    laps: int = 1
    difficulty: int = 2
    max_paths: Optional[int] = 60
    agent_rank_start: Optional[int] = None
    agent_name: str = "RL-Agent"
    agent_use_ai: bool = False
    agent_camera_mode: CameraMode = CameraMode.AUTO  # type: ignore
    with_graphics: bool = False

    def build_agent(self) -> AgentSpec:
        """Return a configured `AgentSpec` instance for environment creation."""
        return AgentSpec(
            rank_start=self.agent_rank_start,
            use_ai=self.agent_use_ai,
            name=self.agent_name,
            camera_mode=self.agent_camera_mode,
        )

    def to_make_kwargs(self) -> dict:
        """Return keyword arguments required by `gym.make()`."""
        kwargs = dict(
            render_mode=self.render_mode,
            track=self.track,
            num_kart=self.num_kart,
            laps=self.laps,
            difficulty=self.difficulty,
        )
        if self.max_paths is not None:
            kwargs["max_paths"] = self.max_paths
        return kwargs


# ---------------------------------------------------------------------------
# Environment Factory
# ---------------------------------------------------------------------------

def unwrapped(env):
    """Recursively unwrap Gymnasium environment to access the base environment."""
    while hasattr(env, "env"):
        env = env.env
    return env

def make_supertux_env(config: Optional[SuperTuxKartEnvConfig] = None) -> gym.Env:
    """
    Create and initialize a SuperTuxKart Gymnasium environment.

    Parameters
    ----------
    config : SuperTuxKartEnvConfig or None
        Optional custom configuration. If omitted, defaults are used.

    Returns
    -------
    gym.Env
        A ready-to-use Gymnasium environment instance.

    Raises
    ------
    ValueError
        If the provided `env_id` is not supported.

    Example
    -------
    >>> env = make_supertux_env()
    >>> obs, info = env.reset()
    """
    config = config or SuperTuxKartEnvConfig()

    if config.env_id not in SUPPORTED_ENV_IDS:
        raise ValueError(
            f"Unknown env_id '{config.env_id}'. "
            f"Valid values: {sorted(SUPPORTED_ENV_IDS)}"
        )
    env = gym.make(
        config.env_id,
        agent=config.build_agent(),
        **config.to_make_kwargs(),
    )

    return env

In [3]:
from Buffers.types import Transition, Batch
config = SuperTuxKartEnvConfig(agent_name="MyModel", with_graphics=False)
env = make_supertux_env(config)
# unwrapped(env).initialize(with_graphics=config.with_graphics)
obs, info = env.reset()
state, reward, terminated, truncated, info = env.step(env.action_space.sample())
transition = Transition(state, reward, terminated, truncated, info)
print(state)
print(reward)
print(terminated)
print(truncated)
print(info)
env.close()
# ix = 0
# done = False
# state, *_ = env.reset()

# while not done:
#     ix += 1
#     action = env.action_space.sample()
#     state, reward, terminated, truncated, _ = env.step(action)
#     done = truncated or terminated

# # Important to stop the STK process
# env.close()

..:: Antarctica Rendering Engine 2.0 ::..
{'phase': 3, 'aux_ticks': array([0.], dtype=float32), 'powerup': 0, 'attachment': 9, 'attachment_time_left': array([0.], dtype=float32), 'max_steer_angle': array([0.42476034], dtype=float32), 'energy': array([0.], dtype=float32), 'skeed_factor': array([1.], dtype=float32), 'shield_time': array([0.], dtype=float32), 'jumping': 0, 'distance_down_track': array([0.], dtype=float32), 'velocity': array([-0.00183476,  0.04728083,  0.07410952], dtype=float32), 'front': array([ 2.9802322e-07, -6.7974319e-04,  7.1849960e-01], dtype=float32), 'center_path_distance': array([3.5722942], dtype=float32), 'center_path': array([3.5559313 , 0.336223  , 0.05994797], dtype=float32), 'items_position': (array([ 4.66933  , -0.2912451, 25.847893 ], dtype=float32), array([ 0.16109812, -0.2912451 , 26.784826  ], dtype=float32), array([21.256979  , -0.29124445, 16.897118  ], dtype=float32), array([-4.3461356 , -0.29124486, 27.831251  ], dtype=float32), array([24.543562  

In [4]:
space = env.observation_space
from gymnasium import spaces

max_key_len = max(len(k) for k in space.keys()) + 1
box_obs = []
sequence_obs = []
for k, v in space.items():
    nk = len(k)
    pad_space = "  " * (max_key_len - nk)
    if isinstance(v, spaces.Box):
        box_obs.append(k)
    elif isinstance(v, spaces.Sequence):
        print(f"{k}{pad_space} : {v}")
        sequence_obs.append(k)

print(f"Box Observations: {box_obs}")
print(f"Sequence Observations: {sequence_obs}")

items_position               : Sequence(Box(-inf, inf, (3,), float32), stack=False)
items_type                       : Sequence(Discrete(7), stack=False)
karts_position               : Sequence(Box(-inf, inf, (3,), float32), stack=False)
paths_distance               : Sequence(Box(0.0, inf, (2,), float32), stack=False)
paths_end                         : Sequence(Box(-inf, inf, (3,), float32), stack=False)
paths_start                     : Sequence(Box(-inf, inf, (3,), float32), stack=False)
paths_width                     : Sequence(Box(0.0, inf, (1,), float32), stack=False)
Box Observations: ['attachment_time_left', 'aux_ticks', 'center_path', 'center_path_distance', 'distance_down_track', 'energy', 'front', 'max_steer_angle', 'shield_time', 'skeed_factor', 'velocity']
Sequence Observations: ['items_position', 'items_type', 'karts_position', 'paths_distance', 'paths_end', 'paths_start', 'paths_width']


In [5]:
space = env.action_space

max_key_len = max(len(k) for k in space.keys()) + 1
print(max_key_len)
discrete_actions = []
continous_actions = []

for k, v in space.items():
    nk = len(k)
    pad_space = "  " * (max_key_len - nk)
    print(f"{k}{pad_space} : {v}")
    if isinstance(v, spaces.Discrete):
        discrete_actions.append(k)
    elif isinstance(v, spaces.Box):
        continous_actions.append(k)
print(f"Discrete Actions: {discrete_actions}")
print(f"Continous Actions: {continous_actions}")

13
acceleration   : Box(0.0, 1.0, (1,), float32)
brake                 : Discrete(2)
drift                 : Discrete(2)
fire                   : Discrete(2)
nitro                 : Discrete(2)
rescue               : Discrete(2)
steer                 : Box(-1.0, 1.0, (1,), float32)
Discrete Actions: ['brake', 'drift', 'fire', 'nitro', 'rescue']
Continous Actions: ['acceleration', 'steer']


In [6]:
from Buffers.episodeBuffer import episodeReplayBuffer
from Buffers.sequenceBuffer import sequenceReplayBuffer
from Buffers.stepBuffer import stepReplayBuffer
from Buffers.types import Transition, Batch
import torch
from torch import device
from Buffers.types import Transition, Batch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# buffer = episodeReplayBuffer(int(5e4), Lmax=50, device=device)
buffer = sequenceReplayBuffer(int(5e4), sequence_length=64, Lmax=50, device=device)
config = SuperTuxKartEnvConfig(agent_name="MyModel", with_graphics=False)
env = make_supertux_env(config)
# unwrapped(env).initialize(with_graphics=config.with_graphics)
states, info = env.reset()
terminated = False
truncated = False

for i in range(3):
    print(f"New Episode {i}")
    states, info = env.reset()
    terminated = False
    truncated = False
    while terminated == False and truncated == False:
        actions = env.action_space.sample()
        next_states, rewards, terminated, truncated, infos  = env.step(actions)
        transition = Transition(
            states=states,
            action=actions,
            next_states=next_states,
            reward=rewards,
            terminated=terminated,
            truncated=truncated,
        )
        buffer.add(transition)
        states = next_states

        if terminated or truncated:
            break
env.close()
   


..:: Antarctica Rendering Engine 2.0 ::..
New Episode 0
New Episode 1
New Episode 2


In [7]:
# box_obs = ['attachment_time_left', 'aux_ticks', 'center_path', 'center_path_distance', 'distance_down_track', 'energy', 'front', 'max_steer_angle', 'shield_time', 'skeed_factor', 'velocity']

# seq_obs = ['items_position', 'items_type', 'karts_position', 'paths_distance', 'paths_end', 'paths_start', 'paths_width']


# discrete_action_keys = ['brake', 'drift', 'fire', 'nitro', 'rescue']
# continuous_action_keys =  ['acceleration', 'steer']

# from utils.buffer_utils import *
# idxs = torch.randint(0, len(buffer.episodes), (2,))
# batches = []
# box_obs = {k: None for k in box_obs}
# seq_obs = {k: None for k in seq_obs}
# seq_mask = {k: None for k in seq_obs}


# for i in idxs:
#     start, length = buffer.episodes[i]
#     traj = buffer.storage[start : start + length]
#     batches.append(traj)

# print(f"number of batches: {len(batches)}")
# keys = batches[0][0].states.keys()
# print(f"keys: {keys}")
# for k in keys:
#     temp_box_obs = []
#     temp_seqs_obs = []
#     temp_seqs_mask = []
#     for transitions in batches:
#         if k in box_obs:
#             a = [torch.tensor(t.states[k]) for t in transitions]
#             # T by D after stacking, where T is a variable length
#             box_obs_padded, mask = pad_temporal_others(a)
#             # box_obs_padded has shape (1500, D)
#             temp_box_obs.append(box_obs_padded)

#         elif k in seq_obs:
#             seq_list = [torch.tensor(t.states[k]).to(device) for t in transitions]
#             padded_seqs, t_mask = pad_seq(seq_list, 50)
#             temp_seqs_obs.append(padded_seqs)
#             temp_seqs_mask.append(t_mask)

#     if k in box_obs:
#         box_obs[k] = torch.stack(temp_box_obs).to(device)
#     elif k in seq_obs:
#         seq_obs[k], seq_mask[k] = pad_temporal(temp_seqs_obs, temp_seqs_mask) 
# print(seq_obs)
# print(seq_mask)




In [8]:
transitions = buffer.storage[10:18]
for k in discrete_actions:
    a = [torch.tensor(t.action[k]) for t in transitions]
    print(f"{k} shape: { torch.stack(a).shape }")
for k in continous_actions:
    a = [torch.tensor(t.action[k]).squeeze() for t in transitions]
    print(f"{k} shape: { torch.stack(a).shape }")

brake shape: torch.Size([8])
drift shape: torch.Size([8])
fire shape: torch.Size([8])
nitro shape: torch.Size([8])
rescue shape: torch.Size([8])
acceleration shape: torch.Size([8])
steer shape: torch.Size([8])


In [9]:
torch.tensor(buffer.storage[0].states['items_position']).shape

/var/folders/hn/qgh15wdx4tg70d4mtjz4c3yc0000gp/T/ipykernel_86938/203688318.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  torch.tensor(buffer.storage[0].states['items_position']).shape


torch.Size([27, 3])

In [10]:
torch.tensor(buffer.storage[0].states['items_type']).shape

torch.Size([27])

In [11]:
buffer.sample(2).box_obs["attachment_time_left"].shape

torch.Size([2, 64, 1])